# Hulk Quantum Bioinformatics Simulator 🧬⚛️💚

Questo notebook è un esperimento didattico che combina **bioinformatica**, **simulazione di mutazioni genetiche** e **quantum computing**.

Il flusso è:

1. download di una sequenza reale da NCBI/GenBank;
2. estrazione della CDS principale;
3. traduzione DNA → proteina tramite codice genetico standard;
4. simulazione di mutazioni da radiazione gamma;
5. codifica della sequenza in feature di codoni o k-mer;
6. encoding quantistico con PennyLane;
7. confronto originale/mutato tramite **quantum fidelity**;
8. calcolo di un indice narrativo chiamato **Hulk Index**.

> Disclaimer: il modello è volutamente fantascientifico e non è un modello radiobiologico predittivo. Le radiazioni ionizzanti non producono superpoteri: producono danno biologico.


## 1. Setup per Google Colab

Su Colab selezionare:

`Runtime → Change runtime type → GPU`

Poi eseguire la cella seguente. Se Colab chiede di riavviare il runtime dopo l'installazione, riavviare e rieseguire le celle dall'inizio.


In [ ]:
!nvidia-smi

!pip -q install --upgrade pip
!pip -q install pennylane biopython numpy matplotlib
!pip -q install "pennylane-lightning[gpu]"


## 2. Test GPU/cuQuantum

Questa cella verifica se `lightning.gpu` è disponibile. Se non lo è, il notebook userà automaticamente `lightning.qubit` su CPU.


In [ ]:
import pennylane as qml
import numpy as np

print("PennyLane version:", qml.__version__)

try:
    test_dev = qml.device("lightning.gpu", wires=2)
    print("OK: lightning.gpu attivo")
except Exception as e:
    print("ATTENZIONE: lightning.gpu non disponibile")
    print("Errore:", e)
    print("Uso fallback CPU: lightning.qubit")
    test_dev = qml.device("lightning.qubit", wires=2)


@qml.qnode(test_dev)
def test_circuit():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.state()


print(test_circuit())


## 3. Configurazione generale

Inserire una email identificativa per le chiamate NCBI Entrez.

Per GitHub la cella è leggibile anche senza esecuzione; per Colab è consigliato impostare una email reale.


In [ ]:
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt

from Bio import Entrez, SeqIO
from Bio.Data import CodonTable

# NCBI richiede una email identificativa per Entrez.
Entrez.email = "your_email@example.com"  # <-- sostituire con la propria email

ACCESSION = "NC_001653.2"  # Hepatitis delta virus complete genome

USE_GPU = True

# Modalità feature quantistica:
# "codon" = 64 feature, 6 qubit
# "kmer"  = 4^K feature, 2*K qubit
#
# K=6 -> 4096 feature  -> 12 qubit
# K=7 -> 16384 feature -> 14 qubit
# K=8 -> 65536 feature -> 16 qubit
QUANTUM_FEATURE_MODE = "kmer"
KMER_K = 8

RANDOM_SEED = 7


## 4. Download da NCBI ed estrazione della CDS

Il notebook usa il record RefSeq `NC_001653.2`, corrispondente al genoma completo di Hepatitis delta virus.


In [ ]:
def fetch_genbank_record(accession):
    """
    Scarica un record GenBank da NCBI.
    Il record contiene sequenza e annotazioni biologiche.
    """
    handle = Entrez.efetch(
        db="nuccore",
        id=accession,
        rettype="gb",
        retmode="text"
    )
    record = SeqIO.read(handle, "genbank")
    handle.close()
    return record


def extract_longest_cds(record):
    """
    Estrae la CDS più lunga dal record GenBank.
    Se non trova CDS, usa l'intera sequenza.
    """
    cds_features = [f for f in record.features if f.type == "CDS"]

    if not cds_features:
        print("Nessuna CDS trovata. Uso l'intera sequenza.")
        seq = str(record.seq).upper().replace("U", "T")
        return seq, "whole_sequence"

    longest = max(cds_features, key=lambda f: len(f.extract(record.seq)))
    cds_seq = str(longest.extract(record.seq)).upper().replace("U", "T")

    product = longest.qualifiers.get("product", ["unknown product"])[0]
    gene = longest.qualifiers.get("gene", ["unknown gene"])[0]

    return cds_seq, f"{gene} / {product}"


## 5. Codice genetico standard

Queste funzioni traducono la sequenza nucleotidica in una sequenza amminoacidica usando la tabella genetica standard.


In [ ]:
STANDARD_CODE = CodonTable.unambiguous_dna_by_id[1]
BASES_DNA = "ACGT"

CODONS = [
    a + b + c
    for a in BASES_DNA
    for b in BASES_DNA
    for c in BASES_DNA
]


def translate_codon(codon):
    """
    Traduce un codone DNA usando il codice genetico standard.
    """
    codon = codon.upper().replace("U", "T")

    if codon in STANDARD_CODE.stop_codons:
        return "*"

    return STANDARD_CODE.forward_table.get(codon, "X")


def translate_sequence(seq):
    """
    Traduce una sequenza nucleotidica in proteina usando il frame 0.
    Mantiene gli stop codon come '*'.
    """
    seq = seq.upper().replace("U", "T")
    usable_len = len(seq) - (len(seq) % 3)
    seq = seq[:usable_len]

    protein = []

    for i in range(0, len(seq), 3):
        codon = seq[i:i + 3]
        protein.append(translate_codon(codon))

    return "".join(protein)


def translate_sequence_strict(seq):
    """
    Traduce una sequenza e si ferma al primo codone di stop.
    Utile per valutare l'effetto di stop prematuri.
    """
    seq = seq.upper().replace("U", "T")
    usable_len = len(seq) - (len(seq) % 3)
    seq = seq[:usable_len]

    protein = []

    for i in range(0, len(seq), 3):
        codon = seq[i:i + 3]
        aa = translate_codon(codon)
        if aa == "*":
            break
        protein.append(aa)

    return "".join(protein)


## 6. Feature bioinformatiche

Sono disponibili due modalità:

- `codon`: vettore di 64 frequenze codoniche → 6 qubit;
- `kmer`: vettore di frequenze k-mer → `2*KMER_K` qubit.

Per sfruttare meglio una GPU/cuQuantum, la modalità `kmer` con `KMER_K = 8` genera 65.536 ampiezze e 16 qubit.


In [ ]:
def codon_frequency_vector(seq, pseudocount=1e-6):
    """
    Vettore di 64 frequenze codoniche.
    Ogni posizione rappresenta un codone.
    """
    seq = seq.upper().replace("U", "T")
    usable_len = len(seq) - (len(seq) % 3)
    seq = seq[:usable_len]

    counts = dict.fromkeys(CODONS, pseudocount)

    for i in range(0, len(seq), 3):
        codon = seq[i:i + 3]
        if codon in counts:
            counts[codon] += 1

    vec = np.array([counts[codon] for codon in CODONS], dtype=float)
    norm = np.linalg.norm(vec)

    if norm == 0:
        raise ValueError("Vettore codonico nullo.")

    return vec / norm


def generate_kmers(k):
    """
    Genera tutti i k-mer DNA in ordine lessicografico A,C,G,T.
    Numero totale: 4^k.
    """
    if k < 1:
        raise ValueError("k deve essere >= 1")

    kmers = [""]

    for _ in range(k):
        kmers = [prefix + base for prefix in kmers for base in BASES_DNA]

    return kmers


def kmer_frequency_vector(seq, k=6, pseudocount=1e-6):
    """
    Vettore normalizzato di frequenze k-mer.
    """
    seq = seq.upper().replace("U", "T")
    seq = "".join([b for b in seq if b in BASES_DNA])

    kmers = generate_kmers(k)
    counts = dict.fromkeys(kmers, pseudocount)

    for i in range(0, len(seq) - k + 1):
        kmer = seq[i:i + k]
        if kmer in counts:
            counts[kmer] += 1

    vec = np.array([counts[kmer] for kmer in kmers], dtype=float)
    norm = np.linalg.norm(vec)

    if norm == 0:
        raise ValueError("Vettore k-mer nullo.")

    return vec / norm


def global_sequence_features(seq, n_angles):
    """
    Calcola feature globali e le trasforma in angoli.
    Restituisce esattamente n_angles valori.
    """
    seq = seq.upper().replace("U", "T")
    seq = "".join([b for b in seq if b in BASES_DNA])

    if len(seq) == 0:
        raise ValueError("Sequenza vuota.")

    gc = (seq.count("G") + seq.count("C")) / len(seq)
    at = (seq.count("A") + seq.count("T")) / len(seq)
    purine = (seq.count("A") + seq.count("G")) / len(seq)
    pyrimidine = (seq.count("C") + seq.count("T")) / len(seq)

    codon_vec = codon_frequency_vector(seq)
    prob = codon_vec ** 2
    codon_entropy = -np.sum(prob * np.log2(prob + 1e-12)) / 6.0

    protein = translate_sequence(seq)
    stop_fraction = protein.count("*") / max(len(protein), 1)
    frame_remainder = (len(seq) % 3) / 2.0

    base_features = np.array([
        gc,
        at,
        purine,
        pyrimidine,
        codon_entropy,
        min(stop_fraction * 10.0, 1.0),
        frame_remainder,
        min(len(seq) / 2000.0, 1.0)
    ])

    base_features = np.clip(base_features, 0.0, 1.0)
    repeated = np.resize(base_features, n_angles)

    return np.pi * repeated


def make_quantum_feature_vector(seq):
    """
    Costruisce il vettore di feature quantistiche.
    """
    if QUANTUM_FEATURE_MODE == "codon":
        vec = codon_frequency_vector(seq)
        n_wires = 6

    elif QUANTUM_FEATURE_MODE == "kmer":
        vec = kmer_frequency_vector(seq, k=KMER_K)
        n_wires = 2 * KMER_K

    else:
        raise ValueError("QUANTUM_FEATURE_MODE deve essere 'codon' oppure 'kmer'.")

    return vec, n_wires


## 7. Modello gamma / Hulk

La funzione seguente simula mutazioni da radiazione gamma in modo probabilistico e volutamente semplificato.


In [ ]:
def gamma_damage_probability(dose_gy, gamma_sensitivity=1.0):
    """
    Probabilità didattica di danno genetico da radiazione gamma.

    Formula saturante:
        p = 1 - exp(-k * dose)
    """
    k = 0.0025 * gamma_sensitivity
    p = 1.0 - np.exp(-k * dose_gy)
    return float(min(0.85, p))


def mutate_gamma_hulk(
    seq,
    dose_gy=80.0,
    gamma_sensitivity=12.0,
    cluster_strength=1.2,
    seed=7
):
    """
    Simula mutazioni fantascientifiche da radiazione gamma.

    Include:
    - sostituzioni puntiformi;
    - bias G -> T;
    - cluster di mutazioni;
    - piccole delezioni;
    - piccole inserzioni;
    - possibili frameshift.
    """
    rng = np.random.default_rng(seed)

    seq = seq.upper().replace("U", "T")
    seq = "".join([b for b in seq if b in BASES_DNA])
    seq = list(seq)

    original = "".join(seq)
    alphabet = np.array(list(BASES_DNA))

    p_damage = gamma_damage_probability(
        dose_gy=dose_gy,
        gamma_sensitivity=gamma_sensitivity
    )

    # Probabilità per base controllata.
    p_sub = min(0.25, p_damage * 0.25)

    # 1. Sostituzioni puntiformi
    for i, b in enumerate(seq):
        if rng.random() < p_sub:
            if b == "G" and rng.random() < 0.50:
                seq[i] = "T"
            else:
                alternatives = alphabet[alphabet != b]
                seq[i] = rng.choice(alternatives)

    # 2. Cluster di danno
    expected_clusters = len(seq) * p_damage * 0.02 * cluster_strength
    n_clusters = rng.poisson(expected_clusters)

    starts = rng.integers(0, max(len(seq), 1), size=n_clusters)
    starts = sorted(starts, reverse=True)

    for start in starts:
        event = rng.choice(
            ["burst_substitution", "deletion", "insertion"],
            p=[0.55, 0.30, 0.15]
        )

        if event == "burst_substitution":
            length = int(rng.integers(2, 8))
            for j in range(start, min(start + length, len(seq))):
                b = seq[j]
                alternatives = alphabet[alphabet != b]
                seq[j] = rng.choice(alternatives)

        elif event == "deletion":
            length = int(rng.integers(1, 6))
            del seq[start:start + length]

        elif event == "insertion":
            length = int(rng.integers(1, 5))
            inserted = "".join(rng.choice(alphabet, size=length))
            seq[start:start] = list(inserted)

    mutated = "".join(seq)

    # Evita casi senza alcuna mutazione in esempi didattici.
    if mutated == original and len(seq) > 0:
        pos = int(rng.integers(0, len(seq)))
        old = seq[pos]
        alternatives = alphabet[alphabet != old]
        seq[pos] = rng.choice(alternatives)
        mutated = "".join(seq)

    return mutated


## 8. Confronto DNA e proteina

Il confronto DNA è posizionale e quindi semplice: in presenza di indel può sovrastimare le differenze. Per un'analisi biologica reale servirebbe un allineamento globale o locale.


In [ ]:
def compare_dna(seq_ref, seq_mut):
    """
    Confronto semplice tra DNA originale e DNA mutato.
    Non è un allineamento biologico completo.
    """
    n = min(len(seq_ref), len(seq_mut))

    differences = sum(
        1 for a, b in zip(seq_ref[:n], seq_mut[:n])
        if a != b
    )

    length_difference = len(seq_mut) - len(seq_ref)

    return {
        "basi_confrontate": n,
        "differenze_posizionali": differences,
        "differenza_lunghezza": length_difference,
        "possibile_indel": length_difference != 0,
        "possibile_frameshift": length_difference % 3 != 0
    }


def compare_protein_simple(prot_ref, prot_mut):
    """
    Confronto semplice posizione-per-posizione tra proteine.
    Se c'è frameshift, servirebbe un vero allineamento.
    """
    n = min(len(prot_ref), len(prot_mut))

    changed = sum(
        1 for a, b in zip(prot_ref[:n], prot_mut[:n])
        if a != b
    )

    new_stops = sum(
        1 for a, b in zip(prot_ref[:n], prot_mut[:n])
        if a != "*" and b == "*"
    )

    return {
        "aa_confrontati": n,
        "aa_cambiati": changed,
        "nuovi_stop": new_stops,
        "differenza_lunghezza_proteina": len(prot_mut) - len(prot_ref)
    }


## 9. Device quantistico e circuito PennyLane

Il circuito codifica il vettore di feature tramite `AmplitudeEmbedding`, applica rotazioni parametrizzate e un pattern semplice di entanglement.


In [ ]:
def make_quantum_device(n_wires):
    """
    Prova a usare lightning.gpu.
    Se fallisce, usa lightning.qubit.
    """
    if USE_GPU:
        try:
            dev = qml.device("lightning.gpu", wires=n_wires)
            print(f"Device quantistico attivo: lightning.gpu, wires={n_wires}")
            return dev
        except Exception as e:
            print("ATTENZIONE: lightning.gpu non disponibile.")
            print("Errore:", e)
            print("Uso fallback CPU: lightning.qubit")

    dev = qml.device("lightning.qubit", wires=n_wires)
    print(f"Device quantistico attivo: lightning.qubit, wires={n_wires}")
    return dev


_dummy_vec, N_WIRES = make_quantum_feature_vector("ATGCGTATGCGT")
dev = make_quantum_device(N_WIRES)


@qml.qnode(dev)
def quantum_genomic_encoder(feature_vec, angles):
    """
    Codifica quantistica della sequenza.
    """
    qml.AmplitudeEmbedding(
        features=feature_vec,
        wires=range(N_WIRES),
        normalize=True
    )

    for w in range(N_WIRES):
        qml.RY(angles[w], wires=w)

    for w in range(N_WIRES - 1):
        qml.CNOT(wires=[w, w + 1])

    for w in range(N_WIRES):
        qml.RZ(0.15 * angles[w], wires=w)

    for w in range(0, N_WIRES - 1, 2):
        qml.CNOT(wires=[w, w + 1])

    return qml.state()


def quantum_similarity(seq_ref, seq_mut):
    """
    Calcola la similarità quantistica tra sequenza originale e mutata.

    Fidelity:
        1.0 = stati identici
        0.0 = stati molto diversi
    """
    vec_ref, n_ref = make_quantum_feature_vector(seq_ref)
    vec_mut, n_mut = make_quantum_feature_vector(seq_mut)

    if n_ref != N_WIRES or n_mut != N_WIRES:
        raise ValueError("Numero di qubit incoerente.")

    angles_ref = global_sequence_features(seq_ref, N_WIRES)
    angles_mut = global_sequence_features(seq_mut, N_WIRES)

    state_ref = quantum_genomic_encoder(vec_ref, angles_ref)
    state_mut = quantum_genomic_encoder(vec_mut, angles_mut)

    fidelity = np.abs(np.vdot(state_ref, state_mut)) ** 2
    return float(fidelity)


## 10. Hulk Index

L'Hulk Index è una metrica narrativa 0–100. Penalizza stop codon prematuri, frameshift e danni estremi, perché troppe mutazioni non significano automaticamente una trasformazione utile.


In [ ]:
def hulk_index(
    dose_gy,
    dna_metrics,
    protein_metrics,
    quantum_fidelity,
    anger_signal=0.9,
    repair_efficiency=0.1
):
    """
    Calcola un indice Hulk da 0 a 100.
    """
    dna_burden = dna_metrics["differenze_posizionali"] / max(
        dna_metrics["basi_confrontate"], 1
    )

    aa_burden = protein_metrics["aa_cambiati"] / max(
        protein_metrics["aa_confrontati"], 1
    )

    quantum_distance = 1.0 - quantum_fidelity

    dose_component = 100.0 * (1.0 - np.exp(-0.03 * dose_gy))
    dna_component = 100.0 * min(1.0, dna_burden / 0.10)
    aa_component = 100.0 * min(1.0, aa_burden / 0.30)
    quantum_component = 100.0 * min(1.0, quantum_distance / 0.50)
    anger_component = 100.0 * np.clip(anger_signal, 0.0, 1.0)

    stop_penalty = 15.0 * protein_metrics["nuovi_stop"]
    frameshift_penalty = 25.0 if dna_metrics["possibile_frameshift"] else 0.0

    adaptation_bonus = 0.0
    if (
        quantum_distance > 0.4
        and not dna_metrics["possibile_frameshift"]
        and protein_metrics["nuovi_stop"] <= 2
        and anger_signal > 0.75
    ):
        adaptation_bonus = 20.0

    raw_score = (
        0.20 * dose_component
        + 0.20 * dna_component
        + 0.25 * aa_component
        + 0.25 * quantum_component
        + 0.10 * anger_component
        + adaptation_bonus
        - stop_penalty
        - frameshift_penalty
    )

    raw_score *= (1.0 - np.clip(repair_efficiency, 0.0, 1.0))
    return float(np.clip(raw_score, 0.0, 100.0))


def classify_hulk(score, dna_metrics=None, protein_metrics=None, quantum_distance=None):
    """
    Classificazione fantascientifica.
    Distingue tra trasformazione compatibile e danno catastrofico.
    """
    if protein_metrics is not None and dna_metrics is not None and quantum_distance is not None:
        aa_burden = protein_metrics["aa_cambiati"] / max(
            protein_metrics["aa_confrontati"], 1
        )

        too_many_stops = protein_metrics["nuovi_stop"] >= 3
        severe_protein_damage = aa_burden > 0.70
        extreme_quantum_shift = quantum_distance > 0.85

        if too_many_stops and severe_protein_damage and extreme_quantum_shift:
            return "GAMMA COLLAPSE - proteina devastata"

        if protein_metrics["nuovi_stop"] >= 6:
            return "danno genetico catastrofico"

        if dna_metrics["possibile_frameshift"] and aa_burden > 0.50:
            return "frameshift gamma instabile"

        if protein_metrics["nuovi_stop"] <= 2 and 0.4 <= quantum_distance <= 0.85 and score >= 25:
            return "She-Hulk candidate"

    if score < 15:
        return "Bruce Banner stabile"
    elif score < 35:
        return "stress cellulare gamma"
    elif score < 55:
        return "mutante gamma instabile"
    elif score < 75:
        return "proto-Hulk"
    elif score < 90:
        return "HULK"
    else:
        return "WORLD BREAKER HULK"


## 11. Modello completo, stampa e grafici


In [ ]:
def run_hulk_model(
    cds_seq,
    dose_gy=80.0,
    gamma_sensitivity=12.0,
    cluster_strength=1.2,
    anger_signal=0.9,
    repair_efficiency=0.1,
    seed=7
):
    """
    Esegue l'intero modello Hulk.
    """
    mut_seq = mutate_gamma_hulk(
        seq=cds_seq,
        dose_gy=dose_gy,
        gamma_sensitivity=gamma_sensitivity,
        cluster_strength=cluster_strength,
        seed=seed
    )

    prot_ref = translate_sequence(cds_seq)
    prot_mut = translate_sequence(mut_seq)

    dna_metrics = compare_dna(cds_seq, mut_seq)
    protein_metrics = compare_protein_simple(prot_ref, prot_mut)

    fidelity = quantum_similarity(cds_seq, mut_seq)
    quantum_distance = 1.0 - fidelity

    score = hulk_index(
        dose_gy=dose_gy,
        dna_metrics=dna_metrics,
        protein_metrics=protein_metrics,
        quantum_fidelity=fidelity,
        anger_signal=anger_signal,
        repair_efficiency=repair_efficiency
    )

    phenotype = classify_hulk(
        score,
        dna_metrics=dna_metrics,
        protein_metrics=protein_metrics,
        quantum_distance=quantum_distance
    )

    return {
        "dose_gy": dose_gy,
        "gamma_sensitivity": gamma_sensitivity,
        "cluster_strength": cluster_strength,
        "anger_signal": anger_signal,
        "repair_efficiency": repair_efficiency,
        "original_length": len(cds_seq),
        "mutated_length": len(mut_seq),
        "dna_metrics": dna_metrics,
        "protein_metrics": protein_metrics,
        "quantum_fidelity": fidelity,
        "quantum_distance": quantum_distance,
        "hulk_index": score,
        "phenotype": phenotype,
        "mutated_sequence": mut_seq,
        "mutated_protein": prot_mut,
        "original_protein": prot_ref,
        "strict_mutated_protein": translate_sequence_strict(mut_seq),
        "strict_original_protein": translate_sequence_strict(cds_seq)
    }


def run_dose_scan(
    cds_seq,
    doses,
    gamma_sensitivity=12.0,
    cluster_strength=1.2,
    anger_signal=0.9,
    repair_efficiency=0.1,
    seed=7
):
    """
    Esegue il modello su più dosi gamma.
    """
    results = []

    for dose in doses:
        result = run_hulk_model(
            cds_seq=cds_seq,
            dose_gy=float(dose),
            gamma_sensitivity=gamma_sensitivity,
            cluster_strength=cluster_strength,
            anger_signal=anger_signal,
            repair_efficiency=repair_efficiency,
            seed=seed
        )
        results.append(result)

    return results


def plot_dose_scan(scan_results):
    """
    Grafici del dose scan.
    """
    doses = [r["dose_gy"] for r in scan_results]
    hulk_scores = [r["hulk_index"] for r in scan_results]
    fidelities = [r["quantum_fidelity"] for r in scan_results]
    quantum_distances = [r["quantum_distance"] for r in scan_results]

    plt.figure(figsize=(8, 5))
    plt.plot(doses, hulk_scores, marker="o")
    plt.xlabel("Dose gamma, Gy")
    plt.ylabel("Hulk Index")
    plt.title("Dose gamma → Hulk Index")
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(doses, fidelities, marker="o")
    plt.xlabel("Dose gamma, Gy")
    plt.ylabel("Quantum fidelity")
    plt.title("Dose gamma → Quantum fidelity")
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(doses, quantum_distances, marker="o")
    plt.xlabel("Dose gamma, Gy")
    plt.ylabel("Quantum distance = 1 - fidelity")
    plt.title("Dose gamma → Quantum distance")
    plt.grid(True)
    plt.show()


def print_result(result):
    """
    Stampa leggibile del risultato singolo.
    """
    print("\n==============================")
    print("        RISULTATO HULK")
    print("==============================")

    print("\nParametri:")
    print("Dose gamma, Gy:", result["dose_gy"])
    print("Sensibilità gamma:", result["gamma_sensitivity"])
    print("Cluster strength:", result["cluster_strength"])
    print("Anger signal:", result["anger_signal"])
    print("DNA repair efficiency:", result["repair_efficiency"])

    print("\nLunghezze:")
    print("Lunghezza originale:", result["original_length"])
    print("Lunghezza mutata:", result["mutated_length"])

    print("\nDNA metrics:")
    for k, v in result["dna_metrics"].items():
        print(k, ":", v)

    print("\nProtein metrics:")
    for k, v in result["protein_metrics"].items():
        print(k, ":", v)

    print("\nQuantum metrics:")
    print("Quantum feature mode:", QUANTUM_FEATURE_MODE)
    print("KMER_K:", KMER_K if QUANTUM_FEATURE_MODE == "kmer" else "non usato")
    print("N_WIRES:", N_WIRES)
    print("Quantum fidelity:", result["quantum_fidelity"])
    print("Quantum distance:", result["quantum_distance"])

    print("\nHULK INDEX:", result["hulk_index"])
    print("Phenotype:", result["phenotype"])

    print("\nPreview sequenza mutata, prime 120 basi:")
    print(result["mutated_sequence"][:120])

    print("\nPreview proteina originale, primi 100 aa:")
    print(result["original_protein"][:100])

    print("\nPreview proteina mutata, primi 100 aa:")
    print(result["mutated_protein"][:100])

    print("\nProteina mutata strict, primi 100 aa:")
    print(result["strict_mutated_protein"][:100])


## 12. Caricamento della sequenza reale


In [ ]:
print("\n==============================")
print(" HULK QUANTUM BIOINFORMATICS")
print("==============================\n")

print("Scarico sequenza reale da NCBI...")
record = fetch_genbank_record(ACCESSION)

print("\nRecord:", record.id)
print("Descrizione:", record.description)
print("Lunghezza genoma:", len(record.seq))

cds_seq, cds_name = extract_longest_cds(record)

print("\nCDS scelta:", cds_name)
print("Lunghezza CDS:", len(cds_seq))
print("Prime 60 basi CDS:")
print(cds_seq[:60])

protein_ref = translate_sequence(cds_seq)

print("\nProteina originale, primi 80 amminoacidi:")
print(protein_ref[:80])

print("\nConfigurazione quantistica:")
print("Feature mode:", QUANTUM_FEATURE_MODE)
print("KMER_K:", KMER_K if QUANTUM_FEATURE_MODE == "kmer" else "non usato")
print("Qubit:", N_WIRES)
print("Dimensione stato:", 2 ** N_WIRES)


## 13. Esperimento: Hulk cinematografico


In [ ]:
result = run_hulk_model(
    cds_seq=cds_seq,
    dose_gy=80.0,
    gamma_sensitivity=12.0,
    cluster_strength=1.2,
    anger_signal=0.90,
    repair_efficiency=0.10,
    seed=7
)

print_result(result)


## 14. Dose scan


In [ ]:
doses = [0, 5, 10, 25, 50, 80, 120, 200]

scan_results = run_dose_scan(
    cds_seq=cds_seq,
    doses=doses,
    gamma_sensitivity=12.0,
    cluster_strength=1.2,
    anger_signal=0.90,
    repair_efficiency=0.10,
    seed=7
)

print("\nDose_Gy | Fidelity | Q_Distance | Hulk_Index | Phenotype")
print("---------------------------------------------------------")

for r in scan_results:
    print(
        f"{r['dose_gy']:7.1f} | "
        f"{r['quantum_fidelity']:8.4f} | "
        f"{r['quantum_distance']:10.4f} | "
        f"{r['hulk_index']:10.2f} | "
        f"{r['phenotype']}"
    )

plot_dose_scan(scan_results)


## 15. Esperimento: Red Hulk / Gamma Collapse


In [ ]:
red_hulk = run_hulk_model(
    cds_seq=cds_seq,
    dose_gy=200.0,
    gamma_sensitivity=25.0,
    cluster_strength=2.0,
    anger_signal=1.0,
    repair_efficiency=0.0,
    seed=42
)

print_result(red_hulk)


## 16. Esperimento: She-Hulk candidate

Parametri più controllati: mutazione moderata, riparazione più alta, niente ricerca di collasso genetico.


In [ ]:
she_hulk = run_hulk_model(
    cds_seq=cds_seq,
    dose_gy=30.0,
    gamma_sensitivity=3.0,
    cluster_strength=0.2,
    anger_signal=0.85,
    repair_efficiency=0.35,
    seed=42
)

print_result(she_hulk)


## Note finali

Questo notebook è pensato per essere letto su GitHub e poi eseguito su Google Colab.

Per esperimenti più leggeri:

```python
QUANTUM_FEATURE_MODE = "codon"
```

Per sfruttare di più la GPU:

```python
QUANTUM_FEATURE_MODE = "kmer"
KMER_K = 8
```

Aumentare `KMER_K` aumenta rapidamente la dimensione dello stato quantistico.
